# Web scraping

In [3]:
import logging
import os

log_path = r'/Users/sanjarbekkakhramonov/Desktop/Python course/5-oy/Project/logs/sraping.log'

os.makedirs(os.path.dirname(log_path), exist_ok=True)

logging.basicConfig(
    filename=log_path,
    filemode='a',
    level=logging.INFO,
    format="%(asctime)s-%(levelname)s-%(message)s"
)

logging.info("Loyiha boshlandi")


In [8]:
import requests
import pandas as pd
import re

try:
    headers = {"User-Agent": "Mozilla/5.0"}

    score_pat = re.compile(r"\d+\s*[–\-—]\s*\d+")
    score_re = re.compile(r"(\d+)\s*[–\-—]\s*(\d+)")

    def extract_score(x: str):
        m = score_re.search(x)
        if not m:
            return None
        return int(m.group(1)), int(m.group(2))

    def scrape_epl_season(start_year):
        end_year = str(start_year + 1)[-2:]
        url = f"https://en.wikipedia.org/wiki/{start_year}%E2%80%93{end_year}_Premier_League"
        print("Scraping:", url)

        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()
        tables = pd.read_html(resp.text)

        # Results jadvalini topish
        cands = []
        for i, t in enumerate(tables):
            try:
                score_cells = t.astype(str).stack().str.contains(score_pat).sum()
                if score_cells > 50:
                    cands.append((i, int(score_cells)))
            except:
                pass

        if not cands:
            print("Results jadvali topilmadi:", start_year)
            return pd.DataFrame()

        best_idx = sorted(cands, key=lambda x: x[1], reverse=True)[0][0]
        results_table = tables[best_idx].copy()

        results_table = results_table.set_index(results_table.columns[0])

        matches = []
        for home_team in results_table.index:
            for away_team in results_table.columns:
                if home_team == away_team:
                    continue

                cell = str(results_table.loc[home_team, away_team])
                score = extract_score(cell)
                if score is None:
                    continue

                hg, ag = score
                matches.append({
                    "season": f"{start_year}-{start_year+1}",
                    "home_team": str(home_team).strip(),
                    "away_team": str(away_team).strip(),
                    "home_goals": hg,
                    "away_goals": ag
                })

        df = pd.DataFrame(matches)
        print("Matches:", df.shape[0])
        return df

    all_dfs = []

    for year in range(2010, 2013):
        season_df = scrape_epl_season(year)
        all_dfs.append(season_df)

    full_df = pd.concat(all_dfs, ignore_index=True)
    print("Total matches:", full_df.shape[0])
    full_df.head()
    logging.info(f"✅ Dataset yuklandi: {full_df.shape}")
except Exception as e:
    logging.exception("🔥 Dataset yuklashda kutilmagan xato!")
    raise

Scraping: https://en.wikipedia.org/wiki/2010%E2%80%9311_Premier_League


/var/folders/28/qwcdyk8x35d4grf02bxmzrf40000gp/T/ipykernel_47933/3455943217.py:24: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)
/var/folders/28/qwcdyk8x35d4grf02bxmzrf40000gp/T/ipykernel_47933/3455943217.py:30: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  score_cells = t.astype(str).stack().str.contains(score_pat).sum()


Matches: 380
Scraping: https://en.wikipedia.org/wiki/2011%E2%80%9312_Premier_League


/var/folders/28/qwcdyk8x35d4grf02bxmzrf40000gp/T/ipykernel_47933/3455943217.py:24: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)
/var/folders/28/qwcdyk8x35d4grf02bxmzrf40000gp/T/ipykernel_47933/3455943217.py:30: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  score_cells = t.astype(str).stack().str.contains(score_pat).sum()


Matches: 380
Scraping: https://en.wikipedia.org/wiki/2012%E2%80%9313_Premier_League
Matches: 380
Total matches: 1140


/var/folders/28/qwcdyk8x35d4grf02bxmzrf40000gp/T/ipykernel_47933/3455943217.py:24: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)
/var/folders/28/qwcdyk8x35d4grf02bxmzrf40000gp/T/ipykernel_47933/3455943217.py:30: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  score_cells = t.astype(str).stack().str.contains(score_pat).sum()


In [9]:
full_df.to_csv("/Users/sanjarbekkakhramonov/Desktop/Python course/5-oy/Project/data/raw/FResults.csv", index=False)
